In [2]:
from pathlib import Path
import sys

INPUT = Path("/kaggle/input")
IMAGE_EXTS = {".png", ".jpg", ".jpeg"}

mura_roots = list(INPUT.rglob("MURA-v1.1"))
if not mura_roots:
    raise FileNotFoundError("Could not find MURA-v1.1 under /kaggle/input")
MURA_ROOT = mura_roots[0]

ra_roots = []
for dataset_dir in INPUT.rglob("dataset"):
    classes = ("Normal", "Mild", "Moderate", "Severe")
    splits_exist = all((dataset_dir / split).is_dir() for split in ("train", "val", "test"))
    classes_exist = all((dataset_dir / "train" / name).is_dir() for name in classes)
    if splits_exist and classes_exist:
        ra_roots.append(dataset_dir)

if not ra_roots:
    raise FileNotFoundError(
        "Could not find the RA dataset with dataset/train, val, test "
        "and Normal/Mild/Moderate/Severe folders."
    )
RA_ROOT = ra_roots[0]

script_roots = [
    p.parent for p in INPUT.rglob("train.py")
    if (p.parent / "models.py").is_file()
    and (p.parent / "preprocess.py").is_file()
]
if not script_roots:
    raise FileNotFoundError(
        "Could not find train.py, models.py, and preprocess.py together. "
        "Attach the RA-CAD-Pipeline-Scripts dataset in Kaggle."
    )
PROJECT_ROOT = script_roots[0]
sys.path.insert(0, str(PROJECT_ROOT))

print("MURA:", MURA_ROOT)
print("RA:", RA_ROOT)
print("Project code:", PROJECT_ROOT)

from train import ManifestDataset
from models import DenseNetBinaryClassifier
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

MURA: /kaggle/input/datasets/cjinny/mura-v11/MURA-v1.1
RA: /kaggle/input/datasets/sindhuvunna/rheumatoid-arthritis-detection/dataset
Project code: /kaggle/input/datasets/thujainanajumuddin/ra-cad-pipeline-scripts
PyTorch: 2.10.0+cu128 | CUDA available: True


In [3]:
import csv

def image_files(folder):
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

def write_manifest(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"{path}: {len(rows)} images")

def make_split(split_name, mura_split):
    rows = []

    # MURA normal controls only; exclude MURA study*_positive images.
    for body_part in ("XR_HAND", "XR_WRIST", "XR_FINGER"):
        body_dir = MURA_ROOT / mura_split / body_part
        if body_dir.exists():
            for negative_dir in body_dir.rglob("study*_negative"):
                rows.extend(
                    {"image_path": str(p), "label": 0}
                    for p in image_files(negative_dir)
                )

    # RA dataset: Normal=0; the three disease categories=1.
    for category in ("Normal", "Mild", "Moderate", "Severe"):
        folder = RA_ROOT / split_name / category
        label = 0 if category == "Normal" else 1
        if folder.exists():
            rows.extend(
                {"image_path": str(p), "label": label}
                for p in image_files(folder)
            )

    return rows

train_rows = make_split("train", "train")
val_rows = make_split("val", "valid")
test_rows = make_split("test", "valid")

if not train_rows or not val_rows or not test_rows:
    raise ValueError(
        f"Empty split detected: train={len(train_rows)}, "
        f"val={len(val_rows)}, test={len(test_rows)}. Check the printed roots."
    )

OUT = Path("/kaggle/working/data")
write_manifest(OUT / "binary_train.csv", train_rows)
write_manifest(OUT / "binary_val.csv", val_rows)
write_manifest(OUT / "binary_test.csv", test_rows)

for name, rows in (("train", train_rows), ("val", val_rows), ("test", test_rows)):
    counts = {label: sum(r["label"] == label for r in rows) for label in (0, 1)}
    print(name, "label counts:", counts)

/kaggle/working/data/binary_train.csv: 17803 images
/kaggle/working/data/binary_val.csv: 1883 images
/kaggle/working/data/binary_test.csv: 1891 images
train label counts: {0: 15807, 1: 1996}
val label counts: {0: 1457, 1: 426}
test label counts: {0: 1459, 1: 432}


In [5]:
def image_files(folder):
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and not p.name.startswith("._")
    )

train_rows = make_split("train", "train")
val_rows = make_split("val", "valid")
test_rows = make_split("test", "valid")

write_manifest(OUT / "binary_train.csv", train_rows)
write_manifest(OUT / "binary_val.csv", val_rows)
write_manifest(OUT / "binary_test.csv", test_rows)

print("Metadata files remaining:",
      sum(Path(r["image_path"]).name.startswith("._") for r in train_rows + val_rows + test_rows))

/kaggle/working/data/binary_train.csv: 17799 images
/kaggle/working/data/binary_val.csv: 1883 images
/kaggle/working/data/binary_test.csv: 1891 images
Metadata files remaining: 0


In [7]:
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
WORKERS = 2

train_ds = ManifestDataset(OUT / "binary_train.csv", "binary", training=True)
val_ds = ManifestDataset(OUT / "binary_val.csv", "binary", training=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=WORKERS, pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=WORKERS, pin_memory=(DEVICE.type == "cuda")
)

model = DenseNetBinaryClassifier(pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Compensate for class imbalance in the training split.
train_labels = np.array([labels[0] for _, labels in train_ds.samples])
n_negative = int((train_labels == 0).sum())
n_positive = int((train_labels == 1).sum())
if not n_negative or not n_positive:
    raise ValueError(f"Training needs both labels; found 0:{n_negative}, 1:{n_positive}")
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([n_negative / n_positive], device=DEVICE)
)

checkpoint_path = Path("/kaggle/working/ra_binary.pt")
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0

    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels[:, 0].float().to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * images.size(0)

    model.eval()
    val_loss_sum = 0.0
    correct = total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels[:, 0].float().to(DEVICE, non_blocking=True)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss_sum += loss.item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
            total += labels.numel()

    train_loss = train_loss_sum / len(train_ds)
    val_loss = val_loss_sum / len(val_ds)
    val_accuracy = correct / max(total, 1)
    print(
        f"Epoch {epoch}/{EPOCHS} | train loss={train_loss:.4f} "
        f"| val loss={val_loss:.4f} | val accuracy={val_accuracy:.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {"state_dict": model.state_dict(), "epoch": epoch,
             "validation_loss": val_loss},
            checkpoint_path
        )
        print("Saved best checkpoint:", checkpoint_path)

print("Training complete. Checkpoint:", checkpoint_path)

Epoch 1/10 | train loss=0.0458 | val loss=0.0067 | val accuracy=0.998
Saved best checkpoint: /kaggle/working/ra_binary.pt
Epoch 2/10 | train loss=0.0076 | val loss=0.0011 | val accuracy=1.000
Saved best checkpoint: /kaggle/working/ra_binary.pt
Epoch 3/10 | train loss=0.0023 | val loss=0.0006 | val accuracy=0.999
Saved best checkpoint: /kaggle/working/ra_binary.pt
Epoch 4/10 | train loss=0.0004 | val loss=0.0002 | val accuracy=1.000
Saved best checkpoint: /kaggle/working/ra_binary.pt
Epoch 5/10 | train loss=0.0190 | val loss=0.0046 | val accuracy=0.999
Epoch 6/10 | train loss=0.0038 | val loss=0.0002 | val accuracy=1.000
Saved best checkpoint: /kaggle/working/ra_binary.pt
Epoch 7/10 | train loss=0.0036 | val loss=0.0008 | val accuracy=1.000
Epoch 8/10 | train loss=0.0143 | val loss=0.0075 | val accuracy=0.998
Epoch 9/10 | train loss=0.0045 | val loss=0.0004 | val accuracy=1.000
Epoch 10/10 | train loss=0.0164 | val loss=0.0063 | val accuracy=0.999
Training complete. Checkpoint: /kaggle/

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

test_ds = ManifestDataset(OUT / "binary_test.csv", "binary", training=False)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=WORKERS, pin_memory=(DEVICE.type == "cuda")
)

saved = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(saved["state_dict"])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        y_true.extend(labels[:, 0].cpu().numpy().tolist())
        y_pred.extend((torch.sigmoid(logits).cpu().numpy() >= 0.5).astype(int).tolist())

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["Normal", "RA"], zero_division=0))

[[1459    0]
 [   0  432]]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      1459
          RA       1.00      1.00      1.00       432

    accuracy                           1.00      1891
   macro avg       1.00      1.00      1.00      1891
weighted avg       1.00      1.00      1.00      1891



In [9]:
from pathlib import Path
p = Path("/kaggle/working/ra_binary.pt")
print(p.exists(), p.stat().st_size if p.exists() else "missing")

True 28436555
